# 04 · CDCL Solver Environment

## Colab setup

In [ ]:
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/neurosat"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)

print("Working folder:", os.getcwd())
print("Files:", os.listdir())

## Imports

In [ ]:
import pickle
import random
import time
import pandas as pd
import matplotlib.pyplot as plt

## Settings

In [ ]:
SEED = 42
CHECK_SIZE = 500
LARGE_CHECK_SIZE = 100

random.seed(SEED)

## Solver file

In [ ]:
%%writefile cdcl.py
import random


class Solver:
    def __init__(self, n_vars, clauses):
        self.n_vars = n_vars
        self.clauses = [list(c) for c in clauses]
        self.n_original = len(clauses)
        self.assignment = {}
        self.level = {}
        self.reason = {}
        self.trail = []
        self.trail_lim = []
        self.activity = {v: 0.0 for v in range(1, n_vars + 1)}
        self.bump = 1.0
        self.decisions = 0
        self.conflicts = 0
        self.lbds = []
        self.result = None

    def value(self, lit):
        v = abs(lit)
        if v not in self.assignment:
            return None
        return self.assignment[v] == (lit > 0)

    def assign(self, lit, reason):
        v = abs(lit)
        self.assignment[v] = lit > 0
        self.level[v] = len(self.trail_lim)
        self.reason[v] = reason
        self.trail.append(lit)

    def decide(self, lit):
        self.trail_lim.append(len(self.trail))
        self.assign(lit, None)

    def propagate(self):
        changed = True
        while changed:
            changed = False
            for clause in self.clauses:
                satisfied = False
                free = 0
                last_free = None
                for lit in clause:
                    val = self.value(lit)
                    if val is True:
                        satisfied = True
                        break
                    if val is None:
                        free += 1
                        last_free = lit
                if satisfied:
                    continue
                if free == 0:
                    return clause
                if free == 1:
                    self.assign(last_free, clause)
                    changed = True
        return None

    def analyze(self, conflict):
        current = len(self.trail_lim)
        seen = set()
        learned = []
        counter = 0
        clause = conflict
        index = len(self.trail) - 1

        while True:
            for lit in clause:
                v = abs(lit)
                if v in seen or self.level[v] == 0:
                    continue
                seen.add(v)
                if self.level[v] == current:
                    counter += 1
                else:
                    learned.append(lit)

            while abs(self.trail[index]) not in seen:
                index -= 1
            lit = self.trail[index]
            index -= 1
            counter -= 1

            if counter == 0:
                learned.append(-lit)
                break

            v = abs(lit)
            clause = [l for l in self.reason[v] if abs(l) != v]

        lower_levels = [self.level[abs(l)] for l in learned[:-1]]
        back_level = max(lower_levels) if lower_levels else 0
        lbd = len(set(self.level[abs(l)] for l in learned))
        return learned, back_level, lbd

    def backtrack(self, level):
        if len(self.trail_lim) <= level:
            return
        start = self.trail_lim[level]
        for lit in self.trail[start:]:
            v = abs(lit)
            del self.assignment[v]
            del self.level[v]
            del self.reason[v]
        self.trail = self.trail[:start]
        self.trail_lim = self.trail_lim[:level]

    def bump_activity(self, clause):
        for lit in clause:
            self.activity[abs(lit)] += self.bump
        self.bump /= 0.95
        if self.bump > 1e100:
            for v in self.activity:
                self.activity[v] *= 1e-100
            self.bump *= 1e-100

    def settle(self):
        while True:
            conflict = self.propagate()
            if conflict is None:
                if len(self.assignment) == self.n_vars:
                    self.result = True
                return
            self.conflicts += 1
            if len(self.trail_lim) == 0:
                self.result = False
                return
            learned, back_level, lbd = self.analyze(conflict)
            self.lbds.append(lbd)
            self.backtrack(back_level)
            self.clauses.append(learned)
            self.bump_activity(learned)

    def start(self):
        self.settle()
        return self.result

    def step(self, lit):
        n_before = len(self.lbds)
        self.decisions += 1
        self.decide(lit)
        self.settle()
        return self.lbds[n_before:]

    def unassigned(self):
        return [v for v in range(1, self.n_vars + 1) if v not in self.assignment]


def vsids_literal(solver):
    v = max(solver.unassigned(), key=lambda x: solver.activity[x])
    return -v


def random_literal(solver):
    v = random.choice(solver.unassigned())
    return v if random.random() < 0.5 else -v


def solve(n_vars, clauses, choose, max_decisions=100000):
    solver = Solver(n_vars, clauses)
    solver.start()
    while solver.result is None and solver.decisions < max_decisions:
        solver.step(choose(solver))
    return solver

## Import solver

In [ ]:
from cdcl import Solver, solve, vsids_literal, random_literal

## Satisfiable example

In [ ]:
sat_example = [[1, 3, -2], [-1, -2], [-3, -2, 1]]

solver = solve(3, sat_example, vsids_literal)

print("Result:    ", solver.result)
print("Assignment:", solver.assignment)
print("Decisions: ", solver.decisions)
print("Conflicts: ", solver.conflicts)
print("LBDs:      ", solver.lbds)

## Unsatisfiable example

In [ ]:
unsat_example = [[1, 2], [-1, 2], [1, -2], [-1, -2]]

solver = solve(2, unsat_example, vsids_literal)

print("Result:          ", solver.result)
print("Decisions:       ", solver.decisions)
print("Conflicts:       ", solver.conflicts)
print("LBDs:            ", solver.lbds)
print("Learned clauses: ", solver.clauses[solver.n_original:])

## Load data

In [ ]:
with open("test.pkl", "rb") as f:
    test_data = pickle.load(f)

with open("test_large.pkl", "rb") as f:
    large_test_data = pickle.load(f)

print("Test:", len(test_data))
for n, data in large_test_data.items():
    print(f"Test n={n}:", len(data))

## Step-by-step solving

In [ ]:
problem = test_data[1]
solver = Solver(problem["n_vars"], problem["clauses"])
solver.start()

print(f"Variables {problem['n_vars']} | clauses {len(problem['clauses'])} | label {problem['label']}")

while solver.result is None:
    lit = vsids_literal(solver)
    lbds = solver.step(lit)
    print(f"decision {solver.decisions:3d} | literal {lit:4d} | new LBDs {lbds} | assigned {len(solver.assignment)}/{solver.n_vars}")

print("Result:", solver.result)

## Helper functions

In [ ]:
def satisfies(clauses, assignment):
    return all(any(assignment.get(abs(lit)) == (lit > 0) for lit in clause) for clause in clauses)


def run(problem, choose):
    start = time.perf_counter()
    solver = solve(problem["n_vars"], problem["clauses"], choose)
    elapsed = time.perf_counter() - start

    valid_model = solver.result is not True or satisfies(problem["clauses"], solver.assignment)

    return {
        "n_vars": problem["n_vars"],
        "label": problem["label"],
        "correct": solver.result == bool(problem["label"]),
        "valid_model": valid_model,
        "decisions": solver.decisions,
        "conflicts": solver.conflicts,
        "lbds": solver.lbds,
        "mean_lbd": sum(solver.lbds) / len(solver.lbds) if solver.lbds else 0,
        "ms": elapsed * 1000,
    }

## Correctness with VSIDS

In [ ]:
sample = test_data[:CHECK_SIZE]
vsids_df = pd.DataFrame([run(p, vsids_literal) for p in sample])

print("Formulas:       ", len(vsids_df))
print("Wrong answers:  ", (~vsids_df["correct"]).sum())
print("Invalid models: ", (~vsids_df["valid_model"]).sum())
print("Mean decisions: ", round(vsids_df["decisions"].mean(), 1))
print("Mean conflicts: ", round(vsids_df["conflicts"].mean(), 1))
print("Mean time (ms): ", round(vsids_df["ms"].mean(), 1))

## Correctness with random branching

In [ ]:
random_df = pd.DataFrame([run(p, random_literal) for p in sample])

print("Formulas:       ", len(random_df))
print("Wrong answers:  ", (~random_df["correct"]).sum())
print("Invalid models: ", (~random_df["valid_model"]).sum())
print("Mean decisions: ", round(random_df["decisions"].mean(), 1))
print("Mean conflicts: ", round(random_df["conflicts"].mean(), 1))
print("Mean time (ms): ", round(random_df["ms"].mean(), 1))

## Larger formulas

In [ ]:
large_rows = []
for n, data in large_test_data.items():
    for p in data[:LARGE_CHECK_SIZE]:
        large_rows.append(run(p, vsids_literal))

large_df = pd.DataFrame(large_rows)

print("Wrong answers: ", (~large_df["correct"]).sum())
print("Invalid models:", (~large_df["valid_model"]).sum())
print(large_df.groupby("n_vars")[["decisions", "conflicts", "mean_lbd", "ms"]].mean().round(2))

## Statistics by formula size

In [ ]:
all_df = pd.concat([vsids_df, large_df], ignore_index=True)
all_df["size"] = pd.cut(all_df["n_vars"], bins=[9, 19, 29, 39, 40, 60, 80], labels=["10–19", "20–29", "30–39", "40", "60", "80"])

size_stats = all_df.groupby("size", observed=True)[["decisions", "conflicts", "mean_lbd", "ms"]].mean().round(2)
print(size_stats)

## Time and conflicts by size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(size_stats.index.astype(str), size_stats["ms"])
axes[0].set_xlabel("Variables")
axes[0].set_ylabel("Mean time per formula (ms)")

axes[1].bar(size_stats.index.astype(str), size_stats["conflicts"])
axes[1].set_xlabel("Variables")
axes[1].set_ylabel("Mean conflicts")

plt.tight_layout()
plt.savefig("cdcl_size_stats.png", dpi=150)
plt.show()

## LBD distribution

In [ ]:
vsids_lbds = [lbd for lbds in vsids_df["lbds"] for lbd in lbds]
random_lbds = [lbd for lbds in random_df["lbds"] for lbd in lbds]
bins = range(1, max(vsids_lbds + random_lbds) + 2)

plt.figure(figsize=(7, 4))
plt.hist(vsids_lbds, bins=bins, alpha=0.6, density=True, label="VSIDS")
plt.hist(random_lbds, bins=bins, alpha=0.6, density=True, label="Random")
plt.xlabel("LBD of learned clause")
plt.ylabel("Share of learned clauses")
plt.legend()
plt.tight_layout()
plt.savefig("cdcl_lbd_distribution.png", dpi=150)
plt.show()

print(f"VSIDS mean LBD:  {sum(vsids_lbds) / len(vsids_lbds):.2f}")
print(f"Random mean LBD: {sum(random_lbds) / len(random_lbds):.2f}")

## Save

In [ ]:
with open("cdcl_check.pkl", "wb") as f:
    pickle.dump({"vsids": vsids_df, "random": random_df, "large": large_df}, f)

print("Saved cdcl.py, cdcl_check.pkl, cdcl_size_stats.png and cdcl_lbd_distribution.png")